# 🚀 Imitation Learning: GAIL for F-16 (Linear Longitudinal Model)

A concise, GitHub-friendly example of GAIL (Generative Adversarial Imitation Learning) for F-16 longitudinal dynamics: generating expert demonstrations, training GAIL, and quick visual verification of angle-of-attack tracking quality.

## 📋 Table of Contents
- [📖 Introduction](#📖-Introduction)
- [📦 Imports](#📦-Imports)
- [🔧 Experiment Parameters](#🔧-Experiment-Parameters)
- [✈️ F-16 Environment Initialization](#✈️-F-16-Environment-Initialization)
- [🎓 Expert Demonstrations (PD)](#🎓-Expert-Demonstrations-(PD))
- [🤖 GAIL Training](#🤖-GAIL-Training)
- [📈 Visualization](#📈-Visualization)
- [💡 Tips](#💡-Tips)
- [📄 License](#📄-License)

## 📖 Introduction
GAIL trains a policy to replicate expert behavior using adversarial training with a discriminator. In this example, the expert is a simple PD controller for angle of attack and pitch rate. Then we train GAIL and verify tracking of α.


In [ ]:
# 📦 Imports
import numpy as np
import torch
import gymnasium as gym
import matplotlib.pyplot as plt

import tensoraerospace.agent.gail.model as gail_model
from tensoraerospace.agent.gail.model import GAIL
from tensoraerospace.agent.pid import PID
from tensoraerospace.utils import generate_time_period, convert_tp_to_sec_tp
from tensoraerospace.signals.standart import unit_step

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gail_model.device = device
gail_model.use_cuda = device.type == "cuda"
print(f"Using device: {device}")

In [ ]:
# 🔧 Experiment parameters
np.random.seed(42)

dt = 0.01  # Discretization
tp = generate_time_period(tn=20, dt=dt)  # Time period
tps = convert_tp_to_sec_tp(tp, dt=dt)
number_time_steps = len(tp)  # Number of time steps
reference_signals = np.reshape(
    unit_step(degree=5, tp=tp, time_step=10, output_rad=True), [1, -1]
)  # Reference signal

/root/.cache/pypoetry/virtualenvs/tensoraerospace-CmFnmCpV-py3.10/lib/python3.10/site-packages/torch/cuda/__init__.py:141: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0
/root/.cache/pypoetry/virtualenvs/tensoraerospace-CmFnmCpV-py3.10/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:159: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/root/.cache/pypoetry/virtualenvs/tensoraerospace-CmFnmCpV-py3.10/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:246: UserWarning: WARN: The reward returned by `step()` must be a float, int, np.integer or np

-133.5645


In [ ]:
# ✈️ Initialize the F-16 environment
# initial_state matches the selected states
initial_state = [[0], [0], [0]]  # theta, alpha, q

env = gym.make(
    'LinearLongitudinalF16-v0',
    number_time_steps=number_time_steps,
    initial_state=initial_state,
    reference_signal=reference_signals,
    use_reward=False,
    state_space=["theta", "alpha", "q"],
    output_space=["theta", "alpha", "q"],
    control_space=["ele"],
    tracking_states=["alpha"],
)

env.reset()


In [ ]:
# 🎓 Expert demonstrations (PD)
kp, kd = 6.0, 1.0

expert_traj = []  # list of [state, action]
xt, info = env.reset()

for step in range(number_time_steps - 2):
    setpoint = reference_signals[0, step]
    alpha = float(xt[1, 0])  # alpha is the 2nd component
    q_rate = float(xt[2, 0])

    u = -kp * (alpha - setpoint) - kd * q_rate
    ut = np.array([[u]], dtype=np.float32)

    expert_traj.append(np.hstack([xt.reshape(-1), ut.reshape(-1)]))

    xt, reward, terminated, truncated, info = env.step(ut)
    if terminated or truncated:
        break

expert_data = np.array(expert_traj, dtype=np.float32)
expert_data.shape


In [ ]:
# 🤖 GAIL training
learning_rate = 3e-3
max_steps_per_update = 20
mini_batch_size = 16
ppo_epochs = 4

agent = GAIL(
    env=env,
    learning_rate=learning_rate,
    max_steps=max_steps_per_update,
    mini_batch_size=mini_batch_size,
    epochs=ppo_epochs,
    data=expert_data,
)
agent.learn(max_frames=5000, max_reward=-1)


In [ ]:
# 📈 Visualization
xt, _ = env.reset()
traj_alpha = []
traj_theta = []
traj_q = []
traj_u = []

for step in range(number_time_steps - 2):
    state_t = torch.from_numpy(xt.reshape(1, -1)).float().to(device)
    with torch.no_grad():
        dist, _ = agent.model(state_t)
        ut_tensor = dist.sample()
        ut = ut_tensor.detach().cpu().numpy()

    xt, _, terminated, truncated, _ = env.step(ut.reshape(1, 1))

    traj_theta.append(float(xt[0, 0]))
    traj_alpha.append(float(xt[1, 0]))
    traj_q.append(float(xt[2, 0]))
    traj_u.append(float(ut.reshape(-1)[0]))

    if terminated or truncated:
        break

fig, axes = plt.subplots(3, 1, figsize=(15, 8), sharex=True)
axes[0].plot(tps[: len(traj_alpha)], np.rad2deg(np.array(traj_alpha)), label="alpha [deg]")
axes[0].plot(tps[: len(traj_alpha)], np.rad2deg(reference_signals[0, : len(traj_alpha)]), '--', label="ref [deg]")
axes[0].set_ylabel("alpha, deg")
axes[0].legend()

axes[1].plot(tps[: len(traj_theta)], np.rad2deg(np.array(traj_theta)), label="theta [deg]")
axes[1].set_ylabel("theta, deg")
axes[1].legend()

axes[2].plot(tps[: len(traj_q)], np.rad2deg(np.array(traj_q)), label="q [deg/s]")
axes[2].plot(tps[: len(traj_u)], np.rad2deg(np.array(traj_u)), label="u [deg]")
axes[2].set_ylabel("q / u")
axes[2].set_xlabel("time, s")
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
# (optional) Alternative compact visualization
# Duplicates the main idea; we keep a single variant for stylistic brevity
pass

## 💡 Tips
- **Convergence**: decrease `learning_rate` or increase `mini_batch_size` if behavior is noisy
- **Demonstrations**: replace PD with `PID` for a higher-quality expert
- **Reproducibility**: fix `np.random.seed(...)` and (optionally) seeds for PyTorch

## 📄 License
This example is distributed under the project license (see LICENSE in the repository root).